In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

Exercises:
- E01: Tune the hyperparameters of the training to beat my best validation loss of 2.2
- E02: I was not careful with the intialization of the network in this video. (1) What is the loss you'd get if the predicted probabilities at initialization were perfectly uniform? What loss do we achieve? (2) Can you tune the initialization to get a starting loss that is much more similar to (1)?
- E03: Read the Bengio et al 2003 paper ([link](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)), implement and try any idea from the paper. Did it work?

### Functions for training

In [4]:
def build_dataset(words, block_size):
  X,Y = [], []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]

  X= torch.tensor(X)
  Y= torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

In [5]:
def initialize_parameters(n_embd, n_hidden, vocab_size, block_size, generator = torch.Generator().manual_seed(2147483647)):
  C = torch.randn((vocab_size, n_embd), generator = generator)
  W1 = torch.randn((n_embd * block_size, n_hidden), generator = generator) * (5/3)/(n_embd * block_size)**0.5
  b1 = torch.randn(n_hidden, generator = generator) * 0.1
  W2 = torch.randn((n_hidden, vocab_size), generator = generator) * 0.1
  b2 = torch.randn(vocab_size, generator = generator) * 0.1
  parameters = [C, W1, b1, W2, b2]
  for p in parameters:
    p.requires_grad = True
  return parameters

In [6]:
def train_nn( X, Y, parameters, iterations, batch_size, learning_rate, print_loss = False):
  C, W1, b1, W2, b2 = parameters
  for i in range(iterations):

    #minibatch construct
    ix = torch.randint(0, X.shape[0], (batch_size,))

    # forward pass
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, emb.shape[1] * emb.shape[2]) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])

    if print_loss:
      print(loss.item())

    #backward pass
    for p in parameters:
      p.grad = None
    loss.backward()
    for p in parameters:
      p.data += -learning_rate * p.grad

In [7]:
def get_loss(X, Y, parameters):
  C, W1, b1, W2, b2 = parameters
  emb = C[X]
  h = torch.tanh(emb.view(-1, emb.shape[1] * emb.shape[2]) @ W1 + b1)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Y)
  return loss.item()

In [8]:
def generate_samples(parameters, n_samples, block_size):
  C, W1, b1, W2, b2 = parameters
  for i in range(n_samples):
    out = []
    context = [0] * block_size
    while True:
      emb = C[torch.tensor([context])]
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim = 1)
      ix = torch.multinomial(probs, num_samples = 1).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break

    print(''.join(itos[i] for i in out))

# E01: Done in aj_nn_vid3.ipynb file

# E02: adding kaiming initialization.

In [9]:
block_size = 3
n_embd = 10
n_hidden = 300
vocab_size = len(stoi)

In [10]:
# create test train dev splits
import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8* len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1], block_size)
Xdev, Ydev = build_dataset(words[n1:n2], block_size)
Xte, Yte = build_dataset(words[n2:], block_size)

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [11]:
parameters = initialize_parameters(n_embd, n_hidden, vocab_size, block_size)

In [12]:
# total parameters
print(f'total number of parameters: {sum(p.nelement() for p in parameters)}')

total number of parameters: 17697


In [13]:
train_nn(Xtr, Ytr, parameters, 50000, 64, 0.1)
print("learning rate = 0.1 for first 50000 iterations")
print(f'train loss: {get_loss(Xtr, Ytr, parameters)}')
print(f'dev loss: {get_loss(Xdev, Ydev, parameters)}')
train_nn(Xtr, Ytr, parameters, 50000, 64, 0.01)
print("learning rate = 0.01 for second 50000 iterations")
print(f'train loss: {get_loss(Xtr, Ytr, parameters)}')
print(f'dev loss: {get_loss(Xdev, Ydev, parameters)}')
train_nn(Xtr, Ytr, parameters, 50000, 64, 0.001)
print("learning rate = 0.001 for first 50000 iterations")
print(f'train loss: {get_loss(Xtr, Ytr, parameters)}')
print(f'dev loss: {get_loss(Xdev, Ydev, parameters)}')

learning rate = 0.1 for first 50000 iterations
train loss: 2.0958945751190186
dev loss: 2.1601171493530273
learning rate = 0.01 for second 50000 iterations
train loss: 2.023348331451416
dev loss: 2.09778094291687
learning rate = 0.001 for first 50000 iterations
train loss: 2.015094041824341
dev loss: 2.09084415435791


In [14]:
print("After 150000 iterations")
print(f'final test loss: {get_loss(Xte, Yte, parameters)}')


After 150000 iterations
final test loss: 2.0890612602233887


# Just by adding kaiming initialization we get a loss of 2.08 which beat 2.2 from andrej's video.

Sampled names below

In [15]:
generate_samples(parameters, 20, block_size)

mura.
read.
jay.
khylorstonstoni.
isa.
dre.
keyson.
lilyn.
lio.
aar.
nockenna.
lyn.
devan.
domontae.
oden.
aubrexxa.
guiden.
kyran.
keliah.
zige.
